In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
!nvidia-smi

Wed Apr 22 04:29:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%writefile vector_add.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        c[i] = a[i] + b[i];
}

int main() {
    int n = 1 << 20; // 1M elements
    size_t size = n * sizeof(int);

    int *h_a, *h_b, *h_c;

    // Allocate host memory
    h_a = (int*)malloc(size);
    h_b = (int*)malloc(size);
    h_c = (int*)malloc(size);

    // Initialize
    for (int i = 0; i < n; i++) {
        h_a[i] = i;
        h_b[i] = i;
    }

    int *d_a, *d_b, *d_c;

    // Allocate device memory
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // Copy to GPU
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    vectorAdd<<<blocks, threads>>>(d_a, d_b, d_c, n);

    cudaDeviceSynchronize();

    // Copy result back
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // Verify result
    for (int i = 0; i < 10; i++) {
        printf("%d + %d = %d\n", h_a[i], h_b[i], h_c[i]);
    }

    // Free memory
    free(h_a); free(h_b); free(h_c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);

    return 0;
}

Writing vector_add.cu


In [4]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [5]:
!./vector_add

0 + 0 = 0
1 + 1 = 2
2 + 2 = 4
3 + 3 = 6
4 + 4 = 8
5 + 5 = 10
6 + 6 = 12
7 + 7 = 14
8 + 8 = 16
9 + 9 = 18


In [6]:
%%writefile matrix.cu

#include <iostream>
#include <chrono>

using namespace std;
using namespace std::chrono;

__global__ void multiply(int *A, int *B, int *C, int M, int N, int K){
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if(row<M && col<K){
    int sum = 0;
    for(int i=0; i<N; i++){
      sum += A[row*N + i] * B[i*K + col];
    }
    C[row*K + col] = sum;
  }
}

void initialize(int *matrix, int rows, int cols){
  for(int i=0; i<rows*cols; i++){
    matrix[i] = rand() % 10;
  }
}

void print(int *matrix, int rows, int cols){
  for(int i=0; i<rows*cols; i++){
    cout << matrix[i] << " ";
    if((i+1)%cols == 0){
      cout << endl;
    }
  }
  cout << endl;
}

int main(){
  int M, N, K;

  cout << "Enter number of rows of first matrix: ";
  cin >> M;
  cout << "Enter number of columns of first matrix (and rows of second matrix): ";
  cin >> N;
  cout << "Enter number of columns of second matrix: ";
  cin >> K;

  int *A, *B, *C;

  A = new int[M*N];
  B = new int[N*K];
  C = new int[M*K];

  initialize(A, M, N);
  initialize(B, N, K);
  cout<<"Matrix A:"<<endl;
  print(A, M, N);
  cout<<"Matrix B:"<<endl;
  print(B, N, K);

  // Sequential multiplication
  auto start = high_resolution_clock::now();
  for(int row=0; row<M; row++){
    for(int col=0; col<K; col++){
      int sum = 0;
      for(int i=0; i<N; i++){
        sum += A[row*N + i] * B[i*K + col];
      }
      C[row*K + col] = sum;
    }
  }
  auto stop = high_resolution_clock::now();
  auto seq_duration = duration_cast<microseconds>(stop - start);

  // Device memory allocation
  int *X, *Y, *Z;
  cudaMalloc((void**)&X, M*N*sizeof(int));
  cudaMalloc((void**)&Y, N*K*sizeof(int));
  cudaMalloc((void**)&Z, M*K*sizeof(int));

  // Copy data from host to device
  cudaMemcpy(X, A, M*N*sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(Y, B, N*K*sizeof(int), cudaMemcpyHostToDevice);

  // Define grid and block dimensions
  dim3 threadsPerBlock(16, 16);
  dim3 numBlocks((K + threadsPerBlock.x - 1) / threadsPerBlock.x, \
                 (M + threadsPerBlock.y - 1) / threadsPerBlock.y);

  cout << "Sequential Multiplication of matrix A and B: \n";
  print(C, M, K);

  // Parallel multiplication
  start = high_resolution_clock::now();
  multiply<<<numBlocks, threadsPerBlock>>>(X, Y, Z, M, N, K);
  cudaMemcpy(C, Z, M * K * sizeof(int), cudaMemcpyDeviceToHost);
  stop = high_resolution_clock::now();
  auto par_duration = duration_cast<microseconds>(stop - start);

  cout << "Parallel Multiplication of matrix A and B: \n";
  print(C, M, K);

  cout << "Sequential Multiplication Time: " << seq_duration.count() << " microseconds" << endl;
  cout << "Parallel Multiplication Time: " << par_duration.count() << " microseconds" << endl;

  delete[] A;
  delete[] B;
  delete[] C;

  cudaFree(X);
  cudaFree(Y);
  cudaFree(Z);

  return 0;
}


Writing matrix.cu


In [7]:
!nvcc -arch=sm_75 matrix.cu -o mat

In [10]:
!./mat

Enter number of rows of first matrix: 2
Enter number of columns of first matrix (and rows of second matrix): 2
Enter number of columns of second matrix: 2
Matrix A:
3 6 
7 5 

Matrix B:
3 5 
6 2 

Sequential Multiplication of matrix A and B: 
45 27 
51 45 

Parallel Multiplication of matrix A and B: 
45 27 
51 45 

Sequential Multiplication Time: 0 microseconds
Parallel Multiplication Time: 155 microseconds
